# Dictionaries, sets, and trustworthy structured data

Represent labeled scientific records, validate schemas, reason about uniqueness, and make lookup
and deduplication policies explicit.

**Lecture 1 · Python Foundations I · CMOR 438 / INDE 577**


## How to use this notebook

**Estimated time:** 40 minutes of core instruction, plus 25–35 minutes of practice and extension.

**Prerequisite:** notebooks 01–03 on objects, text, lists, tuples, mutation, and copying.

Follow **Core** during class. Pause before prediction cells and state whether a lookup should fail,
return a default, or reveal a schema defect. **Practice** includes executable success criteria.
**Extension** sections add production tradeoffs and may be completed after class.

Notebook 05 will teach general iteration. This notebook therefore focuses on representation,
lookup, updates, set algebra, and standard-library operations without hiding the ideas inside loops.


## Learning objectives

By the end of this notebook, you should be able to:

- explain the key-to-value model of a dictionary;
- choose bracket access, `.get`, or explicit membership from the field contract;
- distinguish an absent key, a present `None`, and a present falsy value;
- update and merge mappings while detecting accidental overwrites;
- explain hashability requirements for dictionary keys and set members;
- use sets for uniqueness, membership, and set algebra without relying on order;
- validate required and unexpected fields with set differences;
- inspect nested list/dictionary records in readable stages;
- count categorical observations with `collections.Counter`; and
- diagnose aliasing, shallow-copy, and shared-default failures.


## Why this matters in industry

External records rarely arrive as neat positional tuples. APIs, JSON documents, configuration,
model metadata, and feature stores use labeled fields. Labels improve readability, but they also
introduce schema drift: fields can be absent, renamed, duplicated during a merge, or present with an
unexpected type.

Sets solve a different class of problems: “Which fields are missing?”, “Which samples appear in both
cohorts?”, and “Which categories are new?” A set removes duplicates, but that transformation can also
erase frequency and order.

Reliable structured-data work asks:

1. Which keys are required, optional, or forbidden?
2. What does absence mean, and is it different from `None`?
3. Which source wins when mappings disagree?
4. Is uniqueness a fact of the domain or merely an artifact of deduplication?


## Scientific question and running scenario

A spectroscopy service reports metadata and model-screening output for one experiment. We want to
answer: **Is the record complete enough for review, and which expected sensor channels were
observed?**

The raw record has labeled fields, a nested model record, repeated channel names, and no reviewer:

```text
experiment_id  → " EXP-0042 "
wavelength_nm  → 532.0
score          → 0.873
model          → {name, version}
channels       → ["visible", "uv", "visible"]
reviewer       → None
```

A model score in `[0, 1]` is not automatically a calibrated probability. The notebook preserves the
word **score** and treats its interpretation as a separate scientific contract.


## Professional practice: labeled data still needs a schema

| Data scientist asks | Software engineer asks |
| --- | --- |
| What does each field represent, including units? | Which keys and value types are allowed? |
| Is missing reviewer different from no reviewer field? | How are absent and null values distinguished? |
| Are repeated channels repeated observations or duplicates? | Where is deduplication policy recorded? |
| Can metadata from two sources disagree? | How are merge conflicts detected and resolved? |
| Which cohort differences are scientifically meaningful? | Are set comparisons deterministic and tested? |

A dictionary makes labels available; it does not enforce their spelling, type, unit, or meaning.
Assertions and later validation functions make those expectations executable.


In [ ]:
raw_experiment = {
    "experiment_id": " EXP-0042 ",
    "wavelength_nm": 532.0,
    "score": 0.873,
    "model": {"name": "signal-screen", "version": "1.3.0"},
    "channels": ["visible", "uv", "visible"],
    "reviewer": None,
}

assert isinstance(raw_experiment, dict)
assert raw_experiment["reviewer"] is None
assert len(raw_experiment["channels"]) == 3
raw_experiment


## Core: a dictionary maps unique keys to values

A dictionary is a mutable **mapping**. Each unique key identifies one value. Lookup is by key rather
than numeric position:

```text
"experiment_id" ──> " EXP-0042 "
"wavelength_nm" ──> 532.0
"score"         ──> 0.873
```

Python dictionaries preserve insertion order, but their primary semantic purpose is labeled lookup.
Do not use “the third dictionary value” as a schema. If order itself is the domain model, use a
sequence or store the ordering explicitly.


## Core: construct mappings with explicit, unique keys

Curly braces containing `key: value` pairs create a dictionary. `dict(...)` can construct one from
keyword arguments or key/value pairs. Keys in a literal must be unique in meaning, but Python cannot
warn when the same literal key is written twice: the later value silently wins.

Values may have any type and need not all share one type. That flexibility is useful for records and
also why external dictionaries require validation.


In [ ]:
instrument = {
    "name": "spectrometer-A",
    "wavelength_unit": "nm",
    "calibrated": True,
}

same_instrument = dict(
    name="spectrometer-A",
    wavelength_unit="nm",
    calibrated=True,
)

overwritten_literal = {"status": "raw", "status": "reviewed"}

assert instrument == same_instrument
assert overwritten_literal == {"status": "reviewed"}
assert len(overwritten_literal) == 1


## Core: access strategy expresses the field contract

- `record[key]` returns the value or raises `KeyError`. Use it when absence means invalid data.
- `record.get(key)` returns `None` when absent. Use it only when that result is unambiguous.
- `record.get(key, default)` returns a stated default when absent.
- `key in record` tests key membership without retrieving the value.

`.get` is not universally “safer.” A silent default can conceal schema drift. Failing loudly is often
the correct behavior for a required field.


In [ ]:
experiment_score = raw_experiment["score"]
optional_operator = raw_experiment.get("operator")
display_lab = raw_experiment.get("laboratory", "not supplied")

assert experiment_score == 0.873
assert optional_operator is None
assert display_lab == "not supplied"
assert "score" in raw_experiment
assert "operator" not in raw_experiment

try:
    raw_experiment["operator"]
except KeyError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: absent, null, empty, zero, and false are different states

These records carry different evidence:

```python
{}                       # reviewer field absent
{"reviewer": None}       # field present; no reviewer value
{"reviewer": ""}         # field present; empty text
{"approved": False}      # field present; explicit negative state
```

If `.get("reviewer")` returns `None`, it cannot by itself distinguish the first two states. Check
membership when presence matters. Do not use `record.get(key) or fallback` when zero, `False`, or an
empty collection is valid evidence.


In [ ]:
absent_reviewer = {}
null_reviewer = {"reviewer": None}
empty_reviewer = {"reviewer": ""}

assert absent_reviewer.get("reviewer") is None
assert null_reviewer.get("reviewer") is None
assert "reviewer" not in absent_reviewer
assert "reviewer" in null_reviewer
assert empty_reviewer["reviewer"] == ""

zero_count = {"retries": 0}
assert zero_count.get("retries", 3) == 0


### Practice: predict lookup behavior

Given `sample = {"id": "S-17", "replicates": 0, "notes": None}`, predict:

1. `sample["id"]`
2. `sample.get("replicates", 1)`
3. `sample.get("notes", "missing")`
4. `sample.get("operator", "missing")`
5. `"operator" in sample`
6. the exception from `sample["operator"]`

Explain which operations distinguish a missing key from a present `None`.


In [ ]:
sample = {"id": "S-17", "replicates": 0, "notes": None}

assert sample["id"] == "S-17"
assert sample.get("replicates", 1) == 0
assert sample.get("notes", "missing") is None
assert sample.get("operator", "missing") == "missing"
assert "operator" not in sample

try:
    sample["operator"]
except KeyError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: set differences make simple schema checks readable

Dictionary membership tests keys. Converting `.keys()` to a set enables comparisons against a
declared schema:

- `required - observed` gives missing required fields;
- `observed - allowed` gives unexpected fields;
- `required <= observed` asks whether all required fields are present.

Field presence is only the first layer. A production schema must also validate value types, units,
ranges, relationships, and missingness policies.


In [ ]:
required_fields = {"experiment_id", "wavelength_nm", "score", "model"}
optional_fields = {"channels", "reviewer", "operator"}
allowed_fields = required_fields | optional_fields
observed_fields = set(raw_experiment)

missing_required = required_fields - observed_fields
unexpected_fields = observed_fields - allowed_fields

assert missing_required == set()
assert unexpected_fields == set()
assert required_fields <= observed_fields


## Core: assignment and update mutate a dictionary

`record[key] = value` adds a new key or replaces the existing value. `.update(other)` applies many
assignments and also overwrites conflicts. Mutating methods return `None` by convention.

An overwrite may be correct, but it should not silently erase source evidence. Before combining
instrument metadata with analyst corrections, detect intersecting keys and state which source owns
each field.


In [ ]:
working_metadata = {"status": "raw", "operator": "R. Chen"}
working_metadata["quality_flag"] = "pass"
previous_status = working_metadata["status"]
working_metadata["status"] = "reviewed"

update_result = working_metadata.update({"review_count": 1})

assert previous_status == "raw"
assert working_metadata["status"] == "reviewed"
assert working_metadata["quality_flag"] == "pass"
assert update_result is None


## Core: mapping merge syntax includes a conflict policy

`left | right` creates a new dictionary in Python 3.9 and later. If both mappings contain a key, the
right value wins. `left |= right` mutates the left mapping.

The syntax is concise; the policy is still consequential. Compute `left.keys() & right.keys()` before
merging sources that should not overlap, or compare conflicting values before selecting authority.


In [ ]:
instrument_metadata = {"instrument": "A", "unit": "nm", "status": "raw"}
review_metadata = {"reviewer": "R. Chen", "status": "approved"}

conflicting_fields = instrument_metadata.keys() & review_metadata.keys()
combined_metadata = instrument_metadata | review_metadata

assert conflicting_fields == {"status"}
assert combined_metadata["status"] == "approved"
assert instrument_metadata["status"] == "raw"
assert "reviewer" not in instrument_metadata


## Core: dictionary copies are shallow

`record.copy()` creates a new outer dictionary. Rebinding a top-level value in the copy does not
change the source. Nested lists and dictionaries remain shared objects.

For raw nested data, construct a new canonical record or copy the specific nested structures whose
ownership changes. Applying `deepcopy` everywhere can be expensive and can obscure which pieces were
intended to remain shared.


In [ ]:
copied_experiment = raw_experiment.copy()
copied_experiment["score"] = 0.900

assert copied_experiment is not raw_experiment
assert copied_experiment["score"] == 0.900
assert raw_experiment["score"] == 0.873
assert copied_experiment["model"] is raw_experiment["model"]

copied_experiment["model"]["version"] = "1.3.1"
assert raw_experiment["model"]["version"] == "1.3.1"

# Restore the shared nested fixture after demonstrating the effect.
raw_experiment["model"]["version"] = "1.3.0"


## Core: keys and set members must be hashable

Dictionaries and sets use a hash value to locate members efficiently. A key must be **hashable**:
its hash and equality behavior must remain stable while stored.

Strings, numbers, `None`, and tuples containing only hashable values are commonly hashable. Lists,
dictionaries, and sets are mutable and unhashable. A tuple is not automatically hashable if it
contains an unhashable element.


In [ ]:
measurement_by_coordinate = {(532.0, "nm"): 0.873}
assert measurement_by_coordinate[(532.0, "nm")] == 0.873

try:
    invalid_mapping = {[532.0, "nm"]: 0.873}
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")

try:
    invalid_tuple_key = {(532.0, ["nm"]): 0.873}
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: keys, values, and items expose different views

- `record.keys()` views keys;
- `record.values()` views values;
- `record.items()` views `(key, value)` pairs.

These are dynamic view objects, not independent list snapshots. A later dictionary mutation appears
in the view. Convert to a tuple or list only when a snapshot or positional operation is genuinely
needed. Notebook 05 will iterate over these views directly.


In [ ]:
view_demo = {"a": 1, "b": 2}
key_view = view_demo.keys()
key_snapshot = tuple(key_view)

view_demo["c"] = 3

assert set(key_view) == {"a", "b", "c"}
assert key_snapshot == ("a", "b")
assert tuple(view_demo.values()) == (1, 2, 3)
assert tuple(view_demo.items()) == (("a", 1), ("b", 2), ("c", 3))


## Core: a set represents unique hashable members

A set is mutable, contains no duplicates, and supports fast membership tests and mathematical set
operations. Set order must not carry domain meaning. Even if a displayed order appears stable in one
run, code should not treat it as a sorting contract.

`{}` creates an empty dictionary. Use `set()` for an empty set. A set literal with values uses braces
without colons.


In [ ]:
observed_channels = {"visible", "uv", "visible"}
empty_mapping = {}
empty_set = set()

assert observed_channels == {"uv", "visible"}
assert len(observed_channels) == 2
assert isinstance(empty_mapping, dict)
assert isinstance(empty_set, set)
assert "visible" in observed_channels
assert "infrared" not in observed_channels


## Core: deduplication discards information

Converting a sequence to a set removes repeated values. This may be correct for “which categories
occurred?” It is wrong for “how many times did each category occur?” or “in what order were channels
reported?”

Keep the original sequence when frequency, order, or provenance matters. Give the derived set a name
such as `unique_channels` so the information loss is visible.


In [ ]:
reported_channels = ["visible", "uv", "visible"]
unique_channels = set(reported_channels)

assert reported_channels == ["visible", "uv", "visible"]
assert unique_channels == {"uv", "visible"}
assert len(reported_channels) == 3
assert len(unique_channels) == 2


## Core: set algebra answers comparison questions directly

For sets `A` and `B`:

- `A | B` is the union: in either set;
- `A & B` is the intersection: in both;
- `A - B` is the difference: in `A` but not `B`;
- `A ^ B` is the symmetric difference: in exactly one;
- `A <= B` tests whether `A` is a subset of `B`.

Translate each scientific question into words before choosing an operator. `expected - observed` and
`observed - expected` answer different questions.


In [ ]:
expected_channels = {"uv", "visible", "infrared"}
observed_channels = {"uv", "visible", "fluorescence"}

all_channels = expected_channels | observed_channels
shared_channels = expected_channels & observed_channels
missing_channels = expected_channels - observed_channels
unexpected_channels = observed_channels - expected_channels
disagreements = expected_channels ^ observed_channels

assert all_channels == {"uv", "visible", "infrared", "fluorescence"}
assert shared_channels == {"uv", "visible"}
assert missing_channels == {"infrared"}
assert unexpected_channels == {"fluorescence"}
assert disagreements == {"infrared", "fluorescence"}


## Extension: `frozenset` expresses an immutable set value

`frozenset` supports non-mutating set operations but cannot add or remove members. Because it is
hashable when its members are hashable, it can itself be a dictionary key or set member.

Use it when an unordered collection of unique members is one stable value—for example, a declared
combination of sensor channels. Do not use it when order or repeated membership matters.


In [ ]:
channel_configuration = frozenset({"uv", "visible"})
configuration_status = {channel_configuration: "validated"}

assert configuration_status[frozenset({"visible", "uv"})] == "validated"

try:
    channel_configuration.add("infrared")
except AttributeError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: `Counter` preserves frequency information

`collections.Counter` is a dictionary specialized for counts. Constructing one from an iterable
counts each observed value. Missing keys return zero rather than raising `KeyError`.

Learn the dictionary model first, then use a specialized standard-library type when it states intent
more clearly. Notebook 05 will implement the underlying counting loop before comparing it with this
abstraction.


In [ ]:
from collections import Counter

channel_counts = Counter(reported_channels)

assert channel_counts == {"visible": 2, "uv": 1}
assert channel_counts["visible"] == 2
assert channel_counts["infrared"] == 0
assert channel_counts.most_common(1) == [("visible", 2)]


## Core: nested records combine mappings and sequences

Python structures commonly mirror JSON-shaped data: dictionaries with string keys whose values may
include numbers, strings, Booleans, `None`, lists, and other dictionaries. Traverse one boundary at a
time and give intermediate values meaningful names.

“JSON-like” is not the same as validated JSON. Python tuples, sets, non-string keys, and arbitrary
objects do not map directly to JSON, and loading JSON is deferred to Lecture 2's native-file lesson.


In [ ]:
model_metadata = raw_experiment["model"]
model_name = model_metadata["name"]
model_version = model_metadata["version"]
first_reported_channel = raw_experiment["channels"][0]

assert isinstance(model_metadata, dict)
assert model_name == "signal-screen"
assert model_version == "1.3.0"
assert first_reported_channel == "visible"


## Worked example: validate and canonicalize one experiment record

We will create a new internal record rather than mutate raw evidence:

1. compare observed keys with required and allowed sets;
2. normalize the identifier under its declared case-insensitive policy;
3. validate wavelength and score domains;
4. retain the ordered channel sequence and derive a unique-channel set;
5. preserve the nested model metadata by copying it; and
6. keep the present `None` reviewer distinct from an absent field.

This is explicit notebook code. Lecture 2 will move boundary logic into functions and classes.


In [ ]:
required_fields = {"experiment_id", "wavelength_nm", "score", "model", "channels"}
allowed_fields = required_fields | {"reviewer", "operator"}
observed_fields = set(raw_experiment)

missing_fields = required_fields - observed_fields
extra_fields = observed_fields - allowed_fields

assert missing_fields == set()
assert extra_fields == set()

experiment_key = raw_experiment["experiment_id"].strip().casefold()
wavelength_nm = raw_experiment["wavelength_nm"]
screening_score = raw_experiment["score"]
reported_channels = raw_experiment["channels"].copy()
unique_channels = set(reported_channels)

assert wavelength_nm > 0.0
assert 0.0 <= screening_score <= 1.0

canonical_experiment = {
    "experiment_key": experiment_key,
    "wavelength_nm": wavelength_nm,
    "screening_score": screening_score,
    "model": raw_experiment["model"].copy(),
    "reported_channels": reported_channels,
    "unique_channels": unique_channels,
    "reviewer": raw_experiment.get("reviewer"),
}


In [ ]:
assert canonical_experiment["experiment_key"] == "exp-0042"
assert canonical_experiment["unique_channels"] == {"uv", "visible"}
assert canonical_experiment["reported_channels"] == ["visible", "uv", "visible"]
assert canonical_experiment["reviewer"] is None
assert canonical_experiment["model"] == {
    "name": "signal-screen",
    "version": "1.3.0",
}
assert canonical_experiment["model"] is not raw_experiment["model"]
assert raw_experiment["experiment_id"] == " EXP-0042 "


### Professional check: what this validation does not prove

The checks establish a small structural contract, but they do not prove:

- that the experiment ID refers to a known experiment;
- that wavelength is expressed in nanometers rather than another unit;
- that the model version is approved or reproducible;
- that duplicate channel reports are harmless;
- that the score is calibrated or the threshold is scientifically justified; or
- that `None` reviewer has the same meaning across source systems.

Good validation is layered. Do not call a record “clean” without naming which guarantees were tested.


## Extension: shared mutable defaults can couple unrelated keys

`dict.fromkeys(keys, value)` assigns the **same value object** to every key. This is safe for immutable
defaults such as `None`. It surprises people when the default is a mutable list: a change through one
key appears under every key.

Notebook 05 will construct independent lists with iteration. Until then, build small mappings
explicitly when each key requires a separately owned mutable value.


In [ ]:
shared_lists = dict.fromkeys(("uv", "visible"), [])
shared_lists["uv"].append("EXP-0042")

assert shared_lists["uv"] is shared_lists["visible"]
assert shared_lists == {
    "uv": ["EXP-0042"],
    "visible": ["EXP-0042"],
}

independent_lists = {"uv": [], "visible": []}
independent_lists["uv"].append("EXP-0042")
assert independent_lists == {"uv": ["EXP-0042"], "visible": []}


## Debugging playbook for mappings and sets

Inspect in this order:

1. `type(value)` and `repr(value)` — mapping, set, sequence, or unexpected object?
2. `record.keys()` — which fields are actually present?
3. `required - set(record)` — which required fields are absent?
4. `set(record) - allowed` — which fields are unexpected?
5. `key in record` before `.get` — absent or present with `None`?
6. `left.keys() & right.keys()` — which fields will a merge overwrite?
7. `source is candidate` and nested identity checks — alias or shallow copy?
8. compare original sequence length with derived set length — what information was removed?

Never catch every `KeyError` and continue silently. Decide whether the field is required, optional,
renamed, or invalid, and preserve enough evidence to diagnose the source.


## Practice: guided schema audit

Audit the supplied calibration record under this contract:

- required fields: `instrument_id`, `date`, and `coefficients`;
- optional field: `operator`;
- no other fields are allowed.

Derive `practice_missing` and `practice_unexpected`, retrieve the optional operator with a clear
display default, and preserve the source. The assertions are success criteria.


In [ ]:
practice_calibration = {
    "instrument_id": "SPEC-A",
    "date": "2026-08-20",
    "coefficients": [1.02, -0.01],
    "laboratory": "Abercrombie Lab",
}

practice_required = {"instrument_id", "date", "coefficients"}
practice_allowed = practice_required | {"operator"}
practice_observed = set(practice_calibration)

practice_missing = practice_required - practice_observed
practice_unexpected = practice_observed - practice_allowed
practice_operator = practice_calibration.get("operator", "not supplied")

assert practice_missing == set()
assert practice_unexpected == {"laboratory"}
assert practice_operator == "not supplied"
assert practice_calibration["coefficients"] == [1.02, -0.01]


## Practice: independent cohort comparison

Two research groups report sample IDs. Derive:

- all distinct IDs;
- IDs appearing in both groups;
- IDs exclusive to the training cohort;
- IDs exclusive to the validation cohort; and
- IDs appearing in exactly one cohort.

Do not mutate the source sets. Pass the assertions and explain why these operations do not reveal
duplicate counts within either original dataset.


In [ ]:
training_ids = {"S-01", "S-02", "S-03", "S-05"}
validation_ids = {"S-03", "S-04", "S-05"}

all_ids = training_ids | validation_ids
overlap_ids = training_ids & validation_ids
training_only_ids = training_ids - validation_ids
validation_only_ids = validation_ids - training_ids
exclusive_ids = training_ids ^ validation_ids

assert all_ids == {"S-01", "S-02", "S-03", "S-04", "S-05"}
assert overlap_ids == {"S-03", "S-05"}
assert training_only_ids == {"S-01", "S-02"}
assert validation_only_ids == {"S-04"}
assert exclusive_ids == {"S-01", "S-02", "S-04"}
assert training_ids == {"S-01", "S-02", "S-03", "S-05"}


## Extension: make and defend a record-merging policy

An instrument record and a laboratory database both contain `sample_id`, `timestamp`, `operator`,
and `quality_status`, but some values disagree. A teammate proposes `instrument | laboratory`.

Write a design response addressing:

1. which source is authoritative for each field;
2. whether timestamps share a timezone and precision;
3. how conflicting IDs and statuses will be surfaced;
4. which raw records and provenance will be retained;
5. whether a dictionary remains adequate as the schema grows;
6. which tests prevent new fields from being silently ignored.

The merge operator provides mechanics, not scientific authority.


## Common failure modes

| Symptom | Likely mistake | Better response |
| --- | --- | --- |
| required field quietly becomes default | `.get` hid schema drift | use brackets or validate required keys first |
| missing and null appear identical | membership was not checked | distinguish `key in record` from value lookup |
| metadata changes after copying | nested object is shared | copy or reconstruct the owned nested structure |
| one source silently wins | merge/update overwrote a key | compute conflicts and apply an authority policy |
| empty set behaves like mapping | `{}` created a dictionary | use `set()` |
| output order changes | set order was treated as meaningful | sort only for presentation under an explicit key |
| duplicate frequency disappears | sequence was converted to set | retain source and use `Counter` for counts |
| list cannot be a key/member | mutable objects are unhashable | use a stable tuple/frozenset if meaning permits |
| every dictionary key shares one list | mutable `fromkeys` default reused | create independently owned values |

Convenient defaults and deduplication are transformations. Document their information loss and test
the policies they implement.


## Retrieval practice

Answer without running code:

1. What makes dictionary access different from sequence access?
2. When is bracket access better than `.get`?
3. How do you distinguish an absent field from a present `None`?
4. Why can duplicate keys in a literal or merge be dangerous?
5. What does a shallow dictionary copy leave shared?
6. Why must keys and set members be hashable?
7. Why is `{}` not an empty set?
8. Which set expressions find missing and unexpected fields?
9. What information does set conversion discard?
10. Why is `Counter` more appropriate than a set for frequencies?


## Takeaway and next step

Dictionaries represent labeled lookup; sets represent unique membership. Neither validates meaning
automatically. Distinguish absent from null, validate keys before lookup, detect merge conflicts,
preserve raw nested evidence, and use set algebra only when uniqueness—not order or frequency—is the
question.

Notebook 05 turns these representations into computations with iteration, comprehensions, and lazy
generators.


## Further reading

- [Python mapping types — `dict`](https://docs.python.org/3.12/library/stdtypes.html#mapping-types-dict)
- [Python set types](https://docs.python.org/3.12/library/stdtypes.html#set-types-set-frozenset)
- [Python dictionary tutorial](https://docs.python.org/3.12/tutorial/datastructures.html#dictionaries)
- [Python sets tutorial](https://docs.python.org/3.12/tutorial/datastructures.html#sets)
- [Python `collections.Counter`](https://docs.python.org/3.12/library/collections.html#collections.Counter)
- [Python copy operations](https://docs.python.org/3.12/library/copy.html)
- [PEP 584: dictionary union operators](https://peps.python.org/pep-0584/)

Use the reference documentation to confirm mechanics. Use the domain schema and provenance policy to
decide which mechanics are appropriate.
